# AI-Based Anomaly Detection and Intelligent Monitoring of Chiller Operating Data

- **Student Register Number:** 24BCE0551
- **Track:** Multivariate and Time-Series Detection
- **Assigned Method:** PCA-Based Multivariate Anomaly Detector
- **Research Question:** *Can anomalies be detected from correlated temperature and pressure variables?*
- **Official Dataset:** `data/raw/official_dataset/Analyse Chillers_AI Course.xls`

---
## Project Overview
This notebook implements the complete research pipeline across eight structured phases. The current progress implements **Phase 1 (Dataset Understanding)**, **Phase 2 (Data Quality & EDA)**, and **Phase 3 (Data Preprocessing and Anti-Data-Leakage Protocol)**.

## 1. Environment Setup and Library Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Set visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries successfully imported.')

## 2. Phase 1: Official Dataset Inspection & Discovery
- Raw file format: Excel 97-2004 (`.xls`)
- Primary worksheet: `Log_Chiller_South`
- Header offset: Row index 2 (Excel row 3)

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'official_dataset', 'Analyse Chillers_AI Course.xls')
if not os.path.exists(raw_path):
    raw_path = os.path.join('data', 'raw', 'official_dataset', 'Analyse Chillers_AI Course.xls')

df_raw = pd.read_excel(raw_path, sheet_name='Log_Chiller_South', header=2)
df_raw.columns = [str(c).strip() for c in df_raw.columns]
df_raw['Timestamp'] = pd.to_datetime(df_raw['Date'].dt.strftime('%Y-%m-%d') + ' ' + df_raw['Time'].astype(str))
print(f'Raw dataset loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'Fleet distribution:\n{df_raw["CHILLER TAG"].value_counts()}')

## 3. Phase 2: Data Quality Assessment & EDA Highlights
- Completeness: 100.0% (0 missing values across all columns)
- Duplicate rows: 0 exact duplicate rows
- Correlation analysis confirms strong thermodynamic relationships between temperatures and pressures (e.g. Discharge Temp Comp 1 vs Discharge Pressure Comp 1, $r = +0.9578$).
- Zero-variance features: 0 features with variance=0.

In [ ]:
# Filter dual-compressor operating regime (N = 517)
dual_mask = (df_raw['Running Load % Comp 1'] > 0) & (df_raw['Running Load % Comp 2'] > 0)
df_dual = df_raw[dual_mask].sort_values('Timestamp').reset_index(drop=True)
print(f'Dual-compressor fleet size: {len(df_dual)} observations (5 single-compressor rows excluded)')

## 4. Phase 3: Data Preprocessing and Anti-Data-Leakage Protocol

### 4.1 Feature Identification (28 Core Thermodynamic Features)
We restrict the PCA input space strictly to the 28 thermodynamic features directly addressing the research question.

In [ ]:
temp_features = [
    'Evaporator Water In Temperature Deg C', 'Evaporator Water Out Temperature Deg C',
    'Condenser Water In Temperature Deg C', 'Condenser Water Out Temperature Deg C',
    'Discharge Temperature Deg C Comp 1', 'Discharge Temperature Deg C Comp 2',
    'Suction Temperature Deg C', 'Chiler Lift Temperature Deg C Comp 1',
    'Chiler Lift Temperature Deg C Comp 2', 'Suction Saturation Temperature Deg C',
    'Suction Superheat deg C', 'Evaporator Approach',
    'Discharge Saturation Temperature Deg C Comp 1', 'Discharge Saturation Temperature Deg C Comp 2',
    'Discharge Superheat Deg C Comp 1', 'Discharge Superheat Deg C Comp 2',
    'Condenser Approach Comp 1', 'Condenser Approach Comp 2',
    'Liquid Temperature Deg C Comp 1', 'Liquid Temperature Deg C Comp 2',
    'Oil Temperature Deg C'
]

press_features = [
    'Discharge Pressure Kg /cm 2 Comp 1', 'Discharge Pressure Kg /cm 2 Comp 2',
    'Suction Pressure Kg /cm 2', 'Chiller Lift Pressure Kg /cm 2 Comp 1',
    'Chiller Lift Pressure Kg /cm 2 Comp 2', 'EXV Differential Pressure Kg /cm 2 Comp 1',
    'EXV Differential Pressure Kg /cm 2 Comp 2'
]

core_28 = temp_features + press_features
print(f'Selected {len(core_28)} core features ({len(temp_features)} Temp + {len(press_features)} Press)')

### 4.2 Chronological Partitioning (70% / 15% / 15% by Unique Timestamps)
To prevent temporal data leakage when multiple chillers operate at the same hour, partitioning is computed on **unique timestamps**.

In [ ]:
unique_ts = np.sort(df_dual['Timestamp'].unique())
n_unique = len(unique_ts)
n_train_ts = int(n_unique * 0.70)
n_val_ts = int(n_unique * 0.15)

train_ts = set(unique_ts[:n_train_ts])
val_ts = set(unique_ts[n_train_ts:n_train_ts + n_val_ts])
test_ts = set(unique_ts[n_train_ts + n_val_ts:])

df_train = df_dual[df_dual['Timestamp'].isin(train_ts)].reset_index(drop=True)
df_val = df_dual[df_dual['Timestamp'].isin(val_ts)].reset_index(drop=True)
df_test = df_dual[df_dual['Timestamp'].isin(test_ts)].reset_index(drop=True)

print(f'Train Partition:      {len(df_train)} rows ({100*len(df_train)/len(df_dual):.1f}%) | {df_train["Timestamp"].min()} to {df_train["Timestamp"].max()}')
print(f'Validation Partition: {len(df_val)} rows ({100*len(df_val)/len(df_dual):.1f}%) | {df_val["Timestamp"].min()} to {df_val["Timestamp"].max()}')
print(f'Test Partition:       {len(df_test)} rows ({100*len(df_test)/len(df_dual):.1f}%) | {df_test["Timestamp"].min()} to {df_test["Timestamp"].max()}')

X_train_raw = df_train[core_28].copy()
X_val_raw = df_val[core_28].copy()
X_test_raw = df_test[core_28].copy()

### 4.3 Missing-Value Inspection and Median Imputation
`SimpleImputer(strategy='median')` is fitted **strictly on the training partition**.

In [ ]:
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train_raw)

X_train_imp = pd.DataFrame(imputer.transform(X_train_raw), columns=core_28)
X_val_imp = pd.DataFrame(imputer.transform(X_val_raw), columns=core_28)
X_test_imp = pd.DataFrame(imputer.transform(X_test_raw), columns=core_28)
print('Median imputer successfully fitted on training set and applied to all partitions.')

### 4.4 Training Set Feature Variance & Quality Audit
Verify that no feature is constant in the training split to avoid division by zero during standardization.

In [ ]:
train_vars = X_train_imp.var()
zero_vars = train_vars[train_vars == 0]
print(f'Constant features in training partition: {len(zero_vars)}')
print(f'Lowest training variance: {train_vars.min():.6f} ({train_vars.idxmin()})')

### 4.5 Feature Standardization (StandardScaler)
Fit `StandardScaler` strictly on imputed training data and transform all partitions.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_imp)

X_train_scaled = pd.DataFrame(scaler.transform(X_train_imp), columns=core_28)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_imp), columns=core_28)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=core_28)

print(f'StandardScaler fitted on training partition.')
print(f'Training scaled mean range: [{X_train_scaled.mean().min():.2e}, {X_train_scaled.mean().max():.2e}]')
print(f'Training scaled std range:  [{X_train_scaled.std().min():.4f}, {X_train_scaled.std().max():.4f}]')

### 4.6 Preprocessing Validation & Saved Artifact Verification
Confirm that all saved CSV files and joblib transformer pipelines are present in `data/processed/`.

In [ ]:
processed_dir = os.path.join('..', 'data', 'processed') if os.path.exists(os.path.join('..', 'data', 'processed')) else os.path.join('data', 'processed')
expected_files = [
    'selected_features.json', 'split_manifest.json',
    'train_features.csv', 'validation_features.csv', 'test_features.csv',
    'train_metadata.csv', 'validation_metadata.csv', 'test_metadata.csv',
    'imputer.joblib', 'scaler.joblib'
]

for f in expected_files:
    fpath = os.path.join(processed_dir, f)
    print(f'  - {f:25s}: Exists = {os.path.exists(fpath)}')
print('\nAll Phase 3 artifacts successfully verified.')